# Step 3 of RAG: Retrieving Relevant Context Based on User Query**RAG** (Retrieval-Augmented Generation) pipeline recap:1. Document Loading & Chunking (Step 1)2. Storing chunks in a Vector Database (Step 2)3. **Retrieval — finding the most relevant chunks for a user's question** ← *this notebook*4. Passing retrieved chunks to an LLM to generate the final answerThis notebook is self-contained — it quickly recreates and stores the chunks from Steps 1-2,then focuses entirely on **retrieval**: how to turn a user's question into the right pieces ofcontext, and how to filter out irrelevant results.### What you will learn1. Recap: chunk and store documents in ChromaDB2. Basic retrieval with `query()` and `n_results`3. Understanding distance vs. similarity scores4. Filtering out weak/irrelevant matches with a similarity threshold5. Combining semantic search with metadata filters (`where`)6. Building a reusable `retrieve_context()` function7. Formatting retrieved chunks into an LLM-ready prompt (preview of Step 4)8. Testing retrieval with paraphrased, real-world-style questions

### Setup — Install Required Libraries

In [1]:
# Install the libraries required for this notebook.
# Run this cell in Colab or an internet-enabled environment.
!pip install -q langchain-text-splitters chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

## 1. Recap — Chunk and Store Documents (Steps 1 & 2)Two short documents, split into chunks, stored in ChromaDB with metadata. This mirrors theprevious notebooks in this series, condensed into one cell so this notebook runs standalone.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.""",

    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once."""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder
)

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    collection.add(
        documents=doc_chunks,
        metadatas=[
            {"source": source, "chunk_index": i}
            for i in range(len(doc_chunks))
        ],
        ids=[f"{source}_chunk_{i}" for i in range(len(doc_chunks))]
    )

print("Total chunks stored:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks stored: 13


## 2. Basic Retrieval — `query()` and `n_results`Given a user's question, ChromaDB embeds it and returns the `n_results` closest chunks.

In [3]:
user_query = "How does RAG improve answers from a language model?"

results = collection.query(
    query_texts=[user_query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)

Distance: 0.3244 | Source: ai_overview.txt | Chunk #6
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
------------------------------------------------------------
Distance: 0.6356 | Source: ai_overview.txt | Chunk #7
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
------------------------------------------------------------
Distance: 0.6794 | Source: embeddings_in_practice.txt | Chunk #2
allowing the system to find results that match the meaning of the query, not just its exact words.
------------------------------------------------------------


**Discussion Point:** `n_results` controls how many chunks come back. Too few might miss usefulcontext; too many adds noise (and costs more tokens later when passed to an LLM). `n_results=3` to `5`is a common starting point.

## 3. Understanding Distance vs. SimilarityChromaDB's default distance metric is **cosine distance**, where:$$\text{cosine\_distance} = 1 - \text{cosine\_similarity}$$So a **smaller distance** means **higher similarity** (more relevant). We can convert back to asimilarity score to make results easier to reason about.

In [4]:
def distance_to_similarity(distance):
    return 1 - distance


for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    similarity = distance_to_similarity(distance)
    print(
        f"Similarity: {similarity:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )

Similarity: 0.6756 | Source: ai_overview.txt | Chunk #6
Similarity: 0.3644 | Source: ai_overview.txt | Chunk #7
Similarity: 0.3206 | Source: embeddings_in_practice.txt | Chunk #2


## 4. Filtering Out Weak Matches with a Similarity ThresholdNot every chunk returned is actually relevant — if a user asks something completely unrelated to ourdocuments, ChromaDB will still return its "closest" chunks, even if none of them are a good match.A **similarity threshold** lets us discard weak matches instead of blindly trusting `n_results`.

In [5]:
def retrieve_with_threshold(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.3
):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    filtered = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = distance_to_similarity(distance)

        if similarity >= similarity_threshold:
            filtered.append({
                "text": doc,
                "metadata": meta,
                "similarity": similarity
            })

    return filtered


# A relevant query
relevant_matches = retrieve_with_threshold(
    "What is deep learning?",
    collection,
    similarity_threshold=0.3
)

print(f"Relevant query -> {len(relevant_matches)} chunk(s) passed the threshold\n")

for m in relevant_matches:
    print(
        f"Similarity: {m['similarity']:.4f} | "
        f"{m['text'][:80]}..."
    )

Relevant query -> 4 chunk(s) passed the threshold

Similarity: 0.6346 | Deep Learning takes Machine Learning further by using neural networks with many ...
Similarity: 0.4946 | Machine Learning is a subset of AI where systems learn patterns from data instea...
Similarity: 0.3875 | Artificial Intelligence, or AI, is the field of computer science focused on buil...
Similarity: 0.3682 | understanding language, recognizing images, making decisions, and learning from ...


In [6]:
# An unrelated query - notice fewer (or zero) chunks pass the threshold
unrelated_matches = retrieve_with_threshold(
    "What is the best recipe for chocolate cake?",
    collection,
    similarity_threshold=0.3
)

print(
    f"Unrelated query -> "
    f"{len(unrelated_matches)} chunk(s) passed the threshold"
)

if len(unrelated_matches) == 0:
    print(
        "No relevant context found - a real RAG system should say "
        "'I don't know' here instead of guessing."
    )

Unrelated query -> 0 chunk(s) passed the threshold
No relevant context found - a real RAG system should say 'I don't know' here instead of guessing.


**Discussion Point:** This threshold check is what prevents a RAG system from confidently answeringquestions its documents actually know nothing about — a key technique for reducing hallucination.

## 5. Combining Semantic Search with Metadata Filters (`where`)Sometimes we want to restrict retrieval to a specific source, category, or date range, in additionto searching by meaning.

In [7]:
results = collection.query(
    query_texts=["how does search work with meaning instead of keywords"],
    n_results=2,
    where={"source": "embeddings_in_practice.txt"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Similarity: {distance_to_similarity(distance):.4f} | "
        f"Source: {meta['source']}"
    )
    print(doc)
    print("-" * 60)

Similarity: 0.6371 | Source: embeddings_in_practice.txt
allowing the system to find results that match the meaning of the query, not just its exact words.
------------------------------------------------------------
Similarity: 0.6114 | Source: embeddings_in_practice.txt
Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
------------------------------------------------------------


## 6. Building a Reusable `retrieve_context()` FunctionLet's combine everything above (top-k, threshold, optional filter) into one clean function we canreuse for any query.

In [8]:
def retrieve_context(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.25,
    source_filter=None
):
    query_kwargs = {
        "query_texts": [query],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"]
    }

    if source_filter:
        query_kwargs["where"] = {"source": source_filter}

    results = collection.query(**query_kwargs)

    context_chunks = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = distance_to_similarity(distance)

        if similarity >= similarity_threshold:
            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })

    return context_chunks


context = retrieve_context("What is machine learning?", collection)

for c in context:
    print(c)

{'text': 'Machine Learning is a subset of AI where systems learn patterns from data instead of following\nhardcoded rules. Instead of programming every decision by hand, we feed the system many examples,\nand it learns the underlying pattern on its own.', 'source': 'ai_overview.txt', 'chunk_index': 3, 'similarity': 0.7744}
{'text': 'Deep Learning takes Machine Learning further by using neural networks with many layers. These\nlayered networks can automatically discover complex patterns in large amounts of data, such as', 'source': 'ai_overview.txt', 'chunk_index': 4, 'similarity': 0.5253}
{'text': 'Artificial Intelligence, or AI, is the field of computer science focused on building systems\nthat can perform tasks which normally require human intelligence. These tasks include', 'source': 'ai_overview.txt', 'chunk_index': 1, 'similarity': 0.4885}
{'text': 'Artificial Intelligence: An Overview', 'source': 'ai_overview.txt', 'chunk_index': 0, 'similarity': 0.3746}
{'text': 'understanding l

## 7. Formatting Retrieved Chunks into an LLM-Ready Prompt (Preview of Step 4)The final job of retrieval is to hand off clean, well-labeled context to a language model. Here'show that context typically gets formatted into a prompt.

In [9]:
def format_context_for_llm(query, context_chunks):
    if not context_chunks:
        return (
            f"Question: {query}\n\n"
            "No relevant context was found in the knowledge base."
        )

    context_text = "\n\n".join(
        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"
        for c in context_chunks
    )

    prompt = f"""Answer the question using ONLY the context below.
If the answer isn't in the context, say you don't know.

Context:
{context_text}

Question: {query}

Answer:"""

    return prompt


context = retrieve_context(
    "How does RAG combine search with a language model?",
    collection
)

final_prompt = format_context_for_llm(
    "How does RAG combine search with a language model?",
    context
)

print(final_prompt)

Answer the question using ONLY the context below.
If the answer isn't in the context, say you don't know.

Context:
[Source: ai_overview.txt, chunk 6]
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG

[Source: embeddings_in_practice.txt, chunk 2]
allowing the system to find results that match the meaning of the query, not just its exact words.

[Source: embeddings_in_practice.txt, chunk 1]
Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,

[Source: embeddings_in_practice.txt, chunk 0]
Embeddings and Vector Search in Practice

[Source: ai_overview.txt, chunk 7]
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, gro

**Discussion Point:** This exact `final_prompt` string is what Step 4 would send to a language model(free or paid) to generate the final, grounded answer — completing the RAG pipeline.

## 8. Testing Retrieval with Paraphrased, Real-World-Style QuestionsA good retrieval system should still work even when the user's wording doesn't match the document'swording exactly.

In [10]:
test_queries = [
    "Why do companies use vector databases?",
    "What's the difference between AI and machine learning?",
    "How do chatbots understand differently worded questions?",
]

for q in test_queries:
    print(f"Query: {q}")
    matches = retrieve_context(q, collection, n_results=2)

    if not matches:
        print("   No relevant chunks passed the similarity threshold.")
    else:
        for m in matches:
            print(
                f"   [{m['similarity']}] "
                f"({m['source']}) "
                f"{m['text'][:90]}..."
            )

    print()

Query: Why do companies use vector databases?
   [0.4543] (embeddings_in_practice.txt) Embeddings and Vector Search in Practice...
   [0.4013] (embeddings_in_practice.txt) Vector databases such as ChromaDB store these embeddings and allow fast similarity search ...

Query: What's the difference between AI and machine learning?
   [0.6668] (ai_overview.txt) Artificial Intelligence, or AI, is the field of computer science focused on building syste...
   [0.6572] (ai_overview.txt) Machine Learning is a subset of AI where systems learn patterns from data instead of follo...

Query: How do chatbots understand differently worded questions?
   [0.4263] (embeddings_in_practice.txt) allowing the system to find results that match the meaning of the query, not just its exac...
   [0.3312] (embeddings_in_practice.txt) This is especially powerful for customer support systems, where a user might phrase a ques...



## Key Takeaways (Recap for Students)| Concept | What It Does ||---|---|| `n_results` | How many chunks to retrieve per query || Distance vs. similarity | `similarity = 1 - distance` for cosine distance in ChromaDB || Similarity threshold | Filters out weak/irrelevant matches instead of trusting every result || `where` filter | Restricts retrieval to a specific source/category alongside semantic search || `retrieve_context()` | Reusable function combining top-k, threshold, and optional filtering || Prompt formatting | Turns retrieved chunks into a clean, labeled context block for an LLM |Retrieval is the heart of RAG — get this step right (good chunking + good thresholds) and the finalanswer from Step 4 will be far more accurate and far less likely to hallucinate.### Practice Exercises1. Lower `similarity_threshold` to `0.1` and re-run the "chocolate cake" query — how many chunks pass now?   What does that tell you about picking a good threshold?2. Add a `max_chunks` parameter to `retrieve_context()` that limits the *total combined character length*   of returned chunks (useful for staying under an LLM's context window).3. Add a third document to the collection and confirm `retrieve_context()` can pull chunks from it too.

## Practice Exercise 1 — Lower the Similarity Threshold
Lower the threshold to `0.1` for the unrelated chocolate-cake query and inspect how many chunks are returned.

In [11]:
# A lower threshold is intentionally more permissive.
low_threshold_matches = retrieve_with_threshold(
    "What is the best recipe for chocolate cake?",
    collection,
    n_results=5,
    similarity_threshold=0.1
)

print(
    "Chocolate-cake query with threshold 0.1 -> "
    f"{len(low_threshold_matches)} chunk(s)"
)

for m in low_threshold_matches:
    print(
        f"Similarity: {m['similarity']:.4f} | "
        f"Source: {m['metadata']['source']}"
    )

print(
    "\nTakeaway: lowering the threshold can increase recall, "
    "but it can also allow irrelevant context into the prompt. "
    "The threshold should be chosen using representative queries."
)


Chocolate-cake query with threshold 0.1 -> 0 chunk(s)

Takeaway: lowering the threshold can increase recall, but it can also allow irrelevant context into the prompt. The threshold should be chosen using representative queries.


## Practice Exercise 2 — Limit Total Context Length
Add `max_chars` so retrieval stops adding chunks once the combined text length would exceed the limit.

In [12]:
def retrieve_context(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.25,
    source_filter=None,
    max_chars=None
):
    query_kwargs = {
        "query_texts": [query],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"]
    }

    if source_filter:
        query_kwargs["where"] = {"source": source_filter}

    results = collection.query(**query_kwargs)

    context_chunks = []
    total_chars = 0

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = distance_to_similarity(distance)

        if similarity < similarity_threshold:
            continue

        # Do not add a chunk if it would exceed the character budget.
        if max_chars is not None and total_chars + len(doc) > max_chars:
            continue

        context_chunks.append({
            "text": doc,
            "source": meta["source"],
            "chunk_index": meta["chunk_index"],
            "similarity": round(similarity, 4)
        })
        total_chars += len(doc)

    return context_chunks


limited_context = retrieve_context(
    "What is artificial intelligence and machine learning?",
    collection,
    n_results=5,
    max_chars=500
)

print("Returned chunks:", len(limited_context))
print(
    "Total characters:",
    sum(len(c["text"]) for c in limited_context)
)

for c in limited_context:
    print(
        f"\n[{c['similarity']}] "
        f"{c['source']} / chunk {c['chunk_index']}"
    )
    print(c["text"])


Returned chunks: 3
Total characters: 455

[0.7342] ai_overview.txt / chunk 1
Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include

[0.7082] ai_overview.txt / chunk 3
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

[0.4748] ai_overview.txt / chunk 0
Artificial Intelligence: An Overview


## Practice Exercise 3 — Add a Third Document
Add another document with metadata, then confirm that `retrieve_context()` can retrieve it.

In [13]:
third_document = {
    "rag_applications.txt": """RAG applications use retrieval to provide grounded context
to a language model before it generates an answer. This can be useful in enterprise
search, customer support, document question answering, and knowledge assistants.
The retrieval layer helps the model use information from the application's own documents."""
}

for source, text in third_document.items():
    doc_chunks = splitter.split_text(text)

    # Use upsert so the exercise can be re-run without duplicate-ID errors.
    collection.upsert(
        documents=doc_chunks,
        metadatas=[
            {"source": source, "chunk_index": i}
            for i in range(len(doc_chunks))
        ],
        ids=[
            f"{source}_chunk_{i}"
            for i in range(len(doc_chunks))
        ]
    )

print("Total chunks after third document:", collection.count())

third_doc_matches = retrieve_context(
    "Where can RAG be used for customer support and enterprise search?",
    collection,
    n_results=3,
    similarity_threshold=0.2
)

for match in third_doc_matches:
    print(
        f"[{match['similarity']}] "
        f"{match['source']} / chunk {match['chunk_index']}"
    )
    print(match["text"])
    print("-" * 60)


Total chunks after third document: 15
[0.6489] rag_applications.txt / chunk 0
RAG applications use retrieval to provide grounded context
to a language model before it generates an answer. This can be useful in enterprise
search, customer support, document question answering, and knowledge assistants.
------------------------------------------------------------
[0.5283] ai_overview.txt / chunk 6
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
------------------------------------------------------------
[0.3023] embeddings_in_practice.txt / chunk 3
This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.
------------------------------------------------------------
